# Setup

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd

import cv2

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.v2 as transforms
from torchvision import models

import lightning as L
from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    LearningRateMonitor,
    BasePredictionWriter,
    RichProgressBar,
)
from lightning.pytorch.loggers import CSVLogger
from lightning_fabric.utilities.seed import seed_everything

from sklearn.model_selection import StratifiedKFold

import matplotlib.pyplot as plt

print(f"GPU Available: {torch.cuda.is_available()}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Lightning Version: {L.__version__}")

In [ ]:
FILE_PATH = Path.cwd()
TRAIN_CSV_PATH = FILE_PATH / "train.csv"
TRAIN_IMAGE_PATH = FILE_PATH / "train_images"
TEST_IMAGE_PATH = FILE_PATH / "test_images"
RUNS_DIR = FILE_PATH / "runs"

RUNS_DIR.mkdir(exist_ok=True)

SEED = 42
IMAGE_SIZE = 224
CLASSICAL_FEATURE_SIZE = 160

seed_everything(SEED)

print(f"Working directory: {FILE_PATH}")
print(f"Train CSV path: {TRAIN_CSV_PATH}")
print(f"Train image path: {TRAIN_IMAGE_PATH}")
print(f"Test image path: {TEST_IMAGE_PATH}")

In [ ]:
@dataclass(frozen=True)
class DatasetInfo:
    train_df: pd.DataFrame
    val_df: pd.DataFrame
    class_names: list
    class_to_index: dict


def load_metadata() -> pd.DataFrame:
    df = pd.read_csv(TRAIN_CSV_PATH)
    print(f"Loaded {len(df)} rows of {df['TARGET'].nunique()} classes")
    return df


def get_kfold_splits(
    df: pd.DataFrame, n_splits: int = 5, random_state: int = SEED
) -> list:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    folds = []
    for train_idx, val_idx in skf.split(df, df["TARGET"]):
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]
        folds.append((train_df, val_df))
    return folds


def load_image(filename: str) -> np.ndarray:
    # load and convert to RGB
    return cv2.cvtColor(cv2.imread(str(TRAIN_IMAGE_PATH / filename)), cv2.COLOR_BGR2RGB)


def load_test_image(image_id: str) -> np.ndarray:
    return cv2.cvtColor(
        cv2.imread(str(TEST_IMAGE_PATH / f"{image_id}.jpg")), cv2.COLOR_BGR2RGB
    )


def get_submission_image_ids() -> list:
    return [path.stem for path in sorted(list(Path(TEST_IMAGE_PATH).glob("*.jpg")))]

def compute_class_weights(
    train_df: pd.DataFrame, class_names: list
) -> torch.Tensor:
    # Compute class weights to handle imbalance
    counts = (
        train_df["TARGET"]
        .value_counts()
        .reindex(class_names)
        .to_numpy(dtype=np.float32)
    )
    weights = counts.sum() / (len(class_names) * counts)
    return torch.tensor(weights, dtype=torch.float32)

In [ ]:
def build_neural_transforms(train: bool) -> transforms.Compose:
    if train:
        return transforms.Compose(
            [
                transforms.ToImage(),
                transforms.Resize(
                    (int(IMAGE_SIZE * 1.1), int(IMAGE_SIZE * 1.1)), antialias=True
                ),
                transforms.RandomCrop(IMAGE_SIZE),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=10),
                transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
                transforms.RandAugment(2, 10),
                transforms.ToDtype(torch.float32, scale=True),
                transforms.ColorJitter(
                    brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02
                ),
                transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
            ]
        )
    return transforms.Compose(
        [
            transforms.ToImage(),
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
            transforms.ToDtype(torch.float32, scale=True),
            transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ]
    )

In [ ]:
class ImageDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        class_to_index: dict,
        transform=None,
    ):
        self.df = df.reset_index(drop=True)
        self.class_to_index = class_to_index
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = load_image(row["file_name"])
        if self.transform is not None:
            image = self.transform(image)
        label = torch.tensor(self.class_to_index[row["TARGET"]], dtype=torch.int64)
        return image, label


class TestDataset(Dataset):
    """Dataset for loading test images without labels."""

    def __init__(self, image_ids: Iterable, transform=None):
        self.image_ids = list(image_ids)
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        image = load_test_image(image_id)
        if self.transform is not None:
            image = self.transform(image)
        return image, image_id

In [ ]:
class CNN(nn.Module):
    """ResNet18-based CNN for butterfly/moth classification."""
    def __init__(self, num_classes: int) -> None:
        super().__init__()
        self.cnn = models.resnet18()
        self.cnn.conv1 = nn.Conv2d(
            3, 64, kernel_size=7, stride=1, padding=1, bias=False
        )
        self.cnn.maxpool = nn.Identity()
        self.cnn.fc = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(512, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.cnn(x)

In [ ]:
class ButterflyKFoldDataModule(L.LightningDataModule):
    def __init__(
        self,
        batch_size,
        num_workers,
        n_splits,
        fold_index,
        seed: int = SEED,
    ):
        """Initialize k-fold data module."""
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.n_splits = n_splits
        self.fold_index = fold_index
        self.seed = seed

    def setup(self, stage: str = None):
        """Setup data for training/validation/prediction."""
        df = load_metadata()
        kfold_splits = get_kfold_splits(
            df, n_splits=self.n_splits, random_state=self.seed
        )
        train_df, val_df = kfold_splits[self.fold_index]

        class_names = sorted(df["TARGET"].unique())
        class_to_index = {name: index for index, name in enumerate(class_names)}

        self.class_names = class_names
        self.train_df = train_df
        self.val_df = val_df
        self.class_to_index = class_to_index
        self.class_weights = compute_class_weights(self.train_df, self.class_names)

        print(
            f"[Fold {self.fold_index + 1}/{self.n_splits}] "
            f"Train: {len(train_df)}, Val: {len(val_df)}"
        )

        self.train_dataset = ImageDataset(
            self.train_df, self.class_to_index, build_neural_transforms(train=True)
        )
        self.val_dataset = ImageDataset(
            self.val_df, self.class_to_index, build_neural_transforms(train=False)
        )

        image_ids = get_submission_image_ids()
        self.test_dataset = TestDataset(
            image_ids, transform=build_neural_transforms(train=False)
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def predict_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=1,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

In [ ]:
class CNNButterflyClassifier(L.LightningModule):
    def __init__(
        self,
        num_classes: int = 100,
        learning_rate: float = 1e-3,
        weight_decay: float = 1e-4,
        label_smoothing: float = 0.05,
        class_weights: torch.Tensor = None,
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["class_weights"])
        self.cnn = CNN(num_classes=num_classes)
        self.num_classes = num_classes
        self.lr = learning_rate
        self.wd = weight_decay
        self.ls = label_smoothing

        self.criterion = nn.CrossEntropyLoss(
            weight=class_weights,
            label_smoothing=label_smoothing,
        )

    def forward(self, x):
        return self.cnn(x)

    def training_step(self, batch, _):
        x, y = batch
        out = self(x)
        loss = self.criterion(out, y)

        preds = out.argmax(dim=1)
        accuracy = (preds == y).float().mean()

        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log(
            "train_accuracy", accuracy, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True
        )
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        out = self(x)
        loss = self.criterion(out, y)

        preds = out.argmax(dim=1)
        accuracy = (preds == y).float().mean()

        self.log("val_loss", loss, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("val_accuracy", accuracy, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def predict_step(self, batch, _):
        x, _ = batch
        out = self(x)
        preds = out.argmax(dim=1)
        return preds.cpu().numpy()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=self.lr, weight_decay=self.wd
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer=optimizer, T_max=self.trainer.max_epochs
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_accuracy",
                "interval": "epoch",
                "frequency": 1,
            },
        }

    def load_state_dict(self, state_dict, strict=True):
        """Handle checkpoint loading - criterion.weight is not saved in checkpoint."""
        # Remove keys that aren't part of the model
        state_dict.pop("class_weights", None)
        if "criterion.weight" in state_dict:
            state_dict.pop("criterion.weight", None)
        
        # strict=False since criterion.weight missing
        return super().load_state_dict(state_dict, strict=False)

In [ ]:
class CSVPredictionWriter(BasePredictionWriter):
    # for saving submission.csv after prediction
    def __init__(self, output_path: str, datamodule):
        super().__init__(write_interval="batch")
        self.output_path = output_path
        self.datamodule = datamodule
        self.all_preds = []

    def write_on_batch_end(
        self,
        trainer,
        pl_module,
        predictions,
        batch_indices,
        batch,
        batch_idx,
        dataloader_idx,
    ):
        """Collect predictions from batches."""
        self.all_preds.extend(predictions)

    def on_predict_end(self, trainer, pl_module):
        """Save predictions to CSV."""
        if not self.all_preds:
            return

        try:
            if hasattr(self.datamodule, "setup"):
                self.datamodule.setup("predict")

            image_ids = get_submission_image_ids()
            class_names = self.datamodule.class_names
            preds = [class_names[int(idx)] for idx in self.all_preds]

            submission = pd.DataFrame({"ID": image_ids, "TARGET": preds})
            submission.to_csv(self.output_path, index=False)
            print(f"Saved {len(preds)} predictions to {self.output_path}")
        except Exception as e:
            print(f"[Error] saving predictions: {e}")


class KFoldCSVPredictionWriter(CSVPredictionWriter):
    """CSVPredictionWriter with fold-specific filename handling."""

    def __init__(self, output_path: str, datamodule, fold_index: int = -1):
        super().__init__(output_path, datamodule)
        self.fold_index = fold_index

    def on_predict_end(self, trainer, pl_module):
        if not self.all_preds:
            return

        try:
            if hasattr(self.datamodule, "setup"):
                self.datamodule.setup("predict")

            image_ids = get_submission_image_ids()
            class_names = self.datamodule.class_names
            preds = [class_names[int(idx)] for idx in self.all_preds]

            output_path = Path(self.output_path)
            if self.fold_index >= 0:
                stem = output_path.stem
                output_path = output_path.parent / f"{stem}_fold_{self.fold_index}.csv"

            submission = pd.DataFrame({"ID": image_ids, "TARGET": preds})
            submission.to_csv(output_path, index=False)
            print(f"Saved {len(preds)} predictions to {output_path}")
        except Exception as e:
            print(f"[Error] saving predictions: {e}")

# Configuration

In [ ]:
# K-Fold config
N_SPLITS = 6
BATCH_SIZE = 32
NUM_WORKERS = 4
MAX_EPOCHS = 100
LEARNING_RATE = 3e-3
WEIGHT_DECAY = 1e-2
LABEL_SMOOTHING = 0.1
NUM_CLASSES = 100
RUN_NAME = "cnn_butterflies_kfold"
SUBMISSION_PATH = FILE_PATH / "kfold_predictions.csv"
RESUME_FROM_LAST_CHECKPOINT = True  # resume training from last checkpoint



print(f"Number of Folds: {N_SPLITS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Max Epochs: {MAX_EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Weight Decay: {WEIGHT_DECAY}")
print(f"Label Smoothing: {LABEL_SMOOTHING}")

In [ ]:
df = load_metadata()

print(f"\nDataset Statistics:")
print(f"  Total Samples: {len(df)}")
print(f"  Number of Classes: {df['TARGET'].nunique()}")
print(f"\nTop 10 Classes:")
print(df['TARGET'].value_counts().head(10))

In [ ]:
# Class distribution
class_dist = df['TARGET'].value_counts().sort_index()
plt.bar(range(len(class_dist)), class_dist.values, alpha=0.7)
plt.title('Overall Class Distribution', fontsize=14)
plt.xlabel('Class Index')
plt.ylabel('Count')


plt.tight_layout()
plt.show()

In [ ]:
def find_last_checkpoint(fold_run_dir: Path) -> Path | None:
    """Find the most recent checkpoint in a fold directory.
    
    Looks for 'last.ckpt' which is automatically saved by Lightning,
    otherwise returns the most recently modified checkpoint.
    """
    if not fold_run_dir.exists():
        return None
    
    # Look for last.ckpt first
    last_ckpt = fold_run_dir / "last.ckpt"
    if last_ckpt.exists():
        return last_ckpt
    
    # Otherwise find most recent checkpoint
    ckpt_files = list(fold_run_dir.glob("*.ckpt"))
    if ckpt_files:
        return max(ckpt_files, key=lambda p: p.stat().st_mtime)
    
    return None


def train_fold(fold_idx: int, n_splits: int, resume_from_checkpoint: bool = False) -> tuple:
    print(f"\n{'='*60}")
    print(f"Fold {fold_idx + 1}/{n_splits}")
    print(f"{'='*60}\n")
    
    fold_datamodule = ButterflyKFoldDataModule(
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        n_splits=n_splits,
        fold_index=fold_idx,
        seed=SEED,
    )
    
    fold_run_dir = RUNS_DIR / f"{RUN_NAME}_fold_{fold_idx}"
    fold_run_dir.mkdir(parents=True, exist_ok=True)
    
    ckpt_path = None
    if resume_from_checkpoint:
        ckpt_path = find_last_checkpoint(fold_run_dir)
        if ckpt_path:
            print(f"Found checkpoint: {ckpt_path.name}")
        else:
            print(f"No checkpoint found, training from scratch")
    
    # Setup callbacks
    callbacks = [
        EarlyStopping(
            monitor="val_accuracy", patience=12, mode="max", verbose=True
        ),
        ModelCheckpoint(
            dirpath=fold_run_dir,
            filename="best_cnn",
            monitor="val_accuracy",
            mode="max",
            save_top_k=3,
            save_last=True,
        ),
        LearningRateMonitor(
            logging_interval="step", log_momentum=True, log_weight_decay=True
        ),
        RichProgressBar(),
    ]
    
    trainer = L.Trainer(
        max_epochs=MAX_EPOCHS,
        callbacks=callbacks,
        logger=CSVLogger(
            save_dir=fold_run_dir.parent,
            name=f"{RUN_NAME}_fold_{fold_idx}",
            flush_logs_every_n_steps=10,
        ),
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        # strategy="ddp_notebook"
        devices=1, # 2 on kaggle
        precision="16-mixed",
        enable_progress_bar=True,
    )
    
    fold_datamodule.setup("fit")
    model = CNNButterflyClassifier(
        num_classes=NUM_CLASSES,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        label_smoothing=LABEL_SMOOTHING,
        class_weights=fold_datamodule.class_weights,
    )
    
    trainer.fit(model, fold_datamodule, ckpt_path=ckpt_path)
    
    val_acc = trainer.logged_metrics.get("val_accuracy", 0.0)
    print(f"\nFold {fold_idx + 1} - Validation Accuracy: {val_acc:.4f}")
    
    return val_acc, model, trainer

# Train

In [ ]:
fold_metrics = []
fold_models = []

print(f"Starting K-Fold Cross Validation ({N_SPLITS} Folds)")
torch.set_float32_matmul_precision('medium')

In [ ]:
# Train all folds
for fold_idx in range(5, N_SPLITS):
    try:
        val_acc, model, trainer = train_fold(fold_idx, N_SPLITS, resume_from_checkpoint=RESUME_FROM_LAST_CHECKPOINT
)
        fold_metrics.append(val_acc)
        fold_models.append((fold_idx, model))
    except Exception as e:
        print(f"Error training fold {fold_idx}: {e}")
        fold_metrics.append(0.0)

# Prediction

In [25]:
def predict_from_checkpoint(fold_idx: int, checkpoint_path: str = None, output_csv: str = "predictions.csv"):
    dm = ButterflyKFoldDataModule(BATCH_SIZE, NUM_WORKERS, N_SPLITS, fold_idx, SEED)
    dm.setup("predict")
    
    if not checkpoint_path:
        checkpoint_path = find_last_checkpoint(RUNS_DIR / f"{RUN_NAME}_fold_{fold_idx}")
        if not checkpoint_path:
            print(f"❌ No checkpoint for fold {fold_idx}")
            return None
    
    model = CNNButterflyClassifier.load_from_checkpoint(checkpoint_path, class_weights=dm.class_weights)
    trainer = L.Trainer(accelerator="gpu" if torch.cuda.is_available() else "cpu", devices=1)
    preds = [dm.class_names[int(idx)] for idx in np.concatenate(trainer.predict(model, dm))]
    
    result = pd.DataFrame({"ID": get_submission_image_ids(), "TARGET": preds})
    result.to_csv(FILE_PATH / output_csv, index=False)
    print(f"✓ {output_csv}")
    return result

# predict_from_checkpoint(fold_idx=0)

In [26]:
def get_logits_from_checkpoint(fold_idx: int, checkpoint_path: str = None):
    """Get raw logits from checkpoint for ensemble."""
    dm = ButterflyKFoldDataModule(BATCH_SIZE, NUM_WORKERS, N_SPLITS, fold_idx, SEED)
    dm.setup("predict")
    
    if not checkpoint_path:
        checkpoint_path = find_last_checkpoint(RUNS_DIR / f"{RUN_NAME}_fold_{fold_idx}")
        if not checkpoint_path:
            return None, None, None
    
    model = CNNButterflyClassifier.load_from_checkpoint(checkpoint_path, class_weights=dm.class_weights)
    model.eval()
    
    logits = []
    with torch.no_grad():
        for images, _ in dm.predict_dataloader():
            if torch.cuda.is_available():
                images = images.cuda()
            logits.append(model(images).cpu().numpy())
    
    return np.concatenate(logits), get_submission_image_ids(), dm.class_names

def ensemble_predict(fold_indices: list, checkpoint_paths: list = None, output_csv: str = "ensemble.csv", method: str = "averaging"):
    print(f"Ensembling {len(fold_indices)} folds ({method})")
    
    all_logits, img_ids, names = [], None, None
    for i, fold_idx in enumerate(fold_indices):
        logits, img_ids, names = get_logits_from_checkpoint(fold_idx, checkpoint_paths[i] if checkpoint_paths else None)
        if logits is not None:
            all_logits.append(logits)
    
    if not all_logits:
        print("No logits")
        return None
    
    if method == "averaging":
        preds = np.argmax(np.mean(all_logits, axis=0), axis=1)
    else:  # voting
        all_preds = np.argmax(np.array(all_logits), axis=2)
        preds = np.array([np.bincount(all_preds[:, i]).argmax() for i in range(all_preds.shape[1])])
    
    result = pd.DataFrame({"ID": img_ids, "TARGET": [names[i] for i in preds]})
    result.to_csv(FILE_PATH / output_csv, index=False)
    print(f"{output_csv} output")
    return result

# ensemble_predict(list(range(N_SPLITS)), method="averaging")

In [27]:
# ensemble_pred = ensemble_predict(list(range(N_SPLITS)), method="averaging", output_csv="ensemble_avg.csv")
# ensemble_pred = ensemble_predict(list(range(N_SPLITS)), method="voting", output_csv="ensemble_vote.csv")
# ensemble_pred = ensemble_predict([0, 1, 2, 3], method="averaging")

In [28]:
def apply_neighbor_bias(probabilities_df, prob_boost=0.05):
    # update prediction probabilities based on neighbors
    # apply this with assumption that nbring images are more likely to be same species
    # since in real life, photographs are often taken in batches of the same species
    new_pred = probabilities_df.copy()
    new_pred = new_pred * (1 - 2 * prob_boost)
    
    for idx in range(len(probabilities_df)):
        prev_idx = (idx - 1) % len(probabilities_df)
        next_idx = (idx + 1) % len(probabilities_df)
        
        neighbor_prev_species = probabilities_df.iloc[prev_idx].idxmax()
        neighbor_next_species = probabilities_df.iloc[next_idx].idxmax()
        
        if neighbor_prev_species in new_pred.columns:
            new_pred.loc[idx, neighbor_prev_species] += prob_boost
        if neighbor_next_species in new_pred.columns:
            new_pred.loc[idx, neighbor_next_species] += prob_boost
    
    return new_pred

In [29]:
def ensemble_predict_with_neighbor_bias(fold_indices: list, prob_boost: float, checkpoint_paths: list, output_csv):
    """Do ensemble prediction but with neighbor biasing."""    
    all_logits, img_ids, names = [], None, None
    for i, fold_idx in enumerate(fold_indices):
        logits, img_ids, names = get_logits_from_checkpoint(fold_idx, checkpoint_paths[i] if checkpoint_paths else None)
        if logits is not None:
            all_logits.append(logits)
    
    all_probs = [np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True) for logits in all_logits]
    
    avg_probs = np.mean(all_probs, axis=0)
    
    # Create DataFrame of probabilities
    probs_df = pd.DataFrame(avg_probs, columns=names)
    
    probs_df = apply_neighbor_bias(probs_df, prob_boost=prob_boost)
    
    preds = np.argmax(probs_df.values, axis=1) # final preds
    
    result = pd.DataFrame({
        "ID": img_ids, 
        "TARGET": [names[i] for i in preds]
    })

    result.to_csv(FILE_PATH / output_csv, index=False)
    return result

ensemble_predict_with_neighbor_bias(
    list(range(6)), 
    prob_boost=0.24,
    checkpoint_paths=None,
    output_csv="ensemble_avg_biased_0.24.csv"
)

ensemble_predict_with_neighbor_bias(
    list(range(6)), 
    prob_boost=0.23,
    checkpoint_paths=None,
    output_csv="ensemble_avg_biased_0.23.csv"
)

ensemble_predict_with_neighbor_bias(
    list(range(6)), 
    prob_boost=0.26,
    checkpoint_paths=None,
    output_csv="ensemble_avg_biased_0.26.csv"
)

ensemble_predict_with_neighbor_bias(
    list(range(6)), 
    prob_boost=0.27,
    checkpoint_paths=None,
    output_csv="ensemble_avg_biased_0.27.csv"
)

Loaded 12594 rows of 100 classes
[Fold 1/6] Train: 10495, Val: 2099
Loaded 12594 rows of 100 classes
[Fold 2/6] Train: 10495, Val: 2099


KeyboardInterrupt: 

In [ ]:
def compare_ensemble_methods():
    avg = ensemble_predict(list(range(N_SPLITS)), method="averaging", output_csv="ensemble_avg.csv")
    vote = ensemble_predict(list(range(N_SPLITS)), method="voting", output_csv="ensemble_vote.csv")
    
    if avg is not None and vote is not None:
        diff = (avg["TARGET"] != vote["TARGET"]).sum()
        pct = 100 * diff / len(avg)
        print(f"Methods differ on {diff}/{len(avg)} ({pct:.1f}%)")
        return avg, vote
    return None, None

avg, vote = compare_ensemble_methods()